In [ ]:
import sys
sys.path.append('..')

In [ ]:
import numpy as np
import sympy as sp

from moment_invariant_tools.symbolic_tools import symbolic_SFT_tensors, display_tensor_latex

In [ ]:
rank_set = [1, 2, 3]
(v, Q, T), variables = symbolic_SFT_tensors(names=["v", "q", "t"], rank_set=rank_set)

display_tensor_latex(Q, name="Q")
display_tensor_latex(T, name="T")   

In [ ]:
from sympy import simplify
f1 = simplify(np.einsum('ij, ij ->', Q,Q, optimize=False))
f2 = simplify(np.einsum("ij, ik, jk -> ", Q,Q,Q, optimize=False))
f3 = simplify(np.einsum("ij, ikl, klj ->", Q,T,T, optimize=False))
f4 = simplify(np.einsum("ij,ik, kml, lmj ->", Q,Q,T,T, optimize=False))
f5 = simplify(np.einsum("ij,kl,ijm,klm ->", Q,Q,T,T, optimize=False))

fc = simplify(np.einsum('ijk,lmn,il,jm,kn->', T,T, Q,Q,Q, optimize=False)) 
fx = simplify(np.einsum('ijn,jlm,ikm,kln ->', T, T, T, T, optimize=False))


In [ ]:
# Derivative w.r.t. to Q only 
Q_variables = variables[3:8]
T_variables = variables[8:]  
J_Q = sp.simplify(sp.Matrix([f1, f2, f3, f4, f5, fc]).jacobian(Q_variables))
J_Qf = sp.simplify(sp.Matrix([f1, f2, f3, f4, f5, fx]).jacobian(Q_variables))

In [ ]:

QT_variables = sorted(J_Q.free_symbols, key=str)
J_Q_numeric = sp.lambdify(QT_variables, J_Q, modules="numpy", cse=True, docstring_limit=0,)
J_Qf_numeric = sp.lambdify(QT_variables, J_Qf, modules="numpy", cse=True, docstring_limit=0,)

rng = np.random.default_rng(42)
values = rng.uniform(-3.0, 3.0, size=len(QT_variables))

point = dict(zip(QT_variables, values))

J_value = np.asarray(
    J_Q_numeric(*values),
    dtype=np.float64,
)
J_value_f = np.asarray(
    J_Qf_numeric(*values),
    dtype=np.float64,
) 

rank = np.linalg.matrix_rank(J_value)
rank_f = np.linalg.matrix_rank(J_value_f)
print(f"Jacobian rank: {rank} (full), {rank_f} (f1-f5 only)")


In [ ]:
# Define the manifold 
theta, a, b, c = sp.symbols("theta a b c", real=True)

Q_theta = sp.Matrix([
    a + b * sp.cos(2 * theta),  # q0
    b * sp.sin(2 * theta),      # q1
    0,                          # q2
    a - b * sp.cos(2 * theta),  # q3
    0,                          # q4
])

# Tangent direction of your manifold
dQ_dtheta = Q_theta.diff(theta)


# Substitute Q(theta) into the Jacobian
Q_subs = dict(zip(Q_variables, Q_theta))
T_subs = dict(zip(T_variables, [0] * len(T_variables)))
T_subs[T_variables[0]] = c 
T_subs[T_variables[3]] = -c 


subs = Q_subs | T_subs

In [ ]:
parameter_values = {
    a: 2,
    b: 1,
    c: 3,
    theta: sp.pi / 4,
}

In [ ]:
sp.simplify(sp.Matrix([f1, f2, f3, f4, f5, fx]).subs(subs))

In [ ]:
sp.simplify(sp.Matrix([f1, f2, f3, f4, f5, fc]).subs(subs))

In [ ]:
# Total derivative along the theta direction
dJQ_dtheta = sp.simplify(J_Q.subs(subs) * dQ_dtheta)
dJQ_dtheta

In [ ]:
dJQ_F_dtheta = sp.simplify(J_Qf.subs(subs) * dQ_dtheta) 
dJQ_F_dtheta

In [ ]:
sp.simplify(J_Q.subs(subs))

In [ ]:
J_Q.subs(subs).subs(parameter_values).rank()

In [ ]:

rank_set = [1, 2, 3]
flexible_basis = [
    'a,a->',
    'ab,ab->',
    'ab,bc,ac->',
    'abc,abc->',
    'abc,bcd,aef,def->',
    'abc,bcd,ade,efg,ghi,fhi->',
# 'abc,bcd,ade,efg,fgh,hij,ijk,klm,mno,lno->',
 'a,b,ab->',
 'a,b,ac,bc->',
 'a,b,c,abc->',
 'a,b,acd,bcd->',
 'ab,bcd,acd->',
 'ab,bc,ade,cde->',
 'ab,cd,abe,cde->']

(v, Q, T), variables = symbolic_SFT_tensors(names=["v", "Q", "T"], rank_set=rank_set)

fs = []
for f in flexible_basis:
    terms = f.split('->')[0].split(',')
    ranks = [len(term) for term in terms]
    tensors_in_contraction = len(terms)
    symbolic_tuple = []
    for rank in ranks:
        if rank == 1:
            symbolic_tuple.append(v)
        elif rank == 2:
            symbolic_tuple.append(Q)
        elif rank == 3:
            symbolic_tuple.append(T)
        else: 
            raise ValueError(f"Unsupported rank {rank} in contraction {f}")
    fs.append(np.einsum(f, *symbolic_tuple, optimize=False))
    print(f)
    


In [ ]:
J = sp.Matrix(fs).jacobian(variables)
J.rank()

In [ ]:
def groebner_relations_among(functions, variables):
    """
    Try to find polynomial relations P(u_0, ..., u_{m-1}) = 0
    among the functions.

    Warning: this can be very expensive for many tensor components.
    """
    m = len(functions)
    u = sp.symbols(f"u0:{m}")

    equations = [
        u_i - sp.expand(f_i)
        for u_i, f_i in zip(u, functions)
    ]

    # Put original tensor-component variables first, then u variables.
    # With lex order, this eliminates the tensor variables.
    G = sp.groebner(equations, *variables, *u, order="lex")

    relations = []

    for g in G.polys:
        expr = sp.factor(g.as_expr())

        # Keep only expressions involving u variables, not tensor components.
        if not any(expr.has(v) for v in variables):
            relations.append(expr)

    return u, relations, G 

In [ ]:
u, relations, G = groebner_relations_among(functions, variables)

print("u variables:")
print(u)

print("Relations:")
for rel in relations:
   sp.pprint(rel)

In [ ]:
import matplotlib.pyplot as plt
from notebooks.contraction_formats import graph_from_formula, einsum_to_formula
from notebooks.contraction_visualization import draw_graph_example
f1_einsum = "ijk, ijl, kmn, lmn -> "
f2_einsum = "imk, ijl, kjn, lmn -> "
f3_einsum = "ijk, ijk-> "

f1 = np.einsum(f1_einsum, T, T, T, T, optimize=False) 
f2 = np.einsum(f2_einsum, T, T, T, T, optimize=False)
f3 = np.einsum(f3_einsum, T, T, optimize=False)

functions_A = [f1, f2]
functions_B = [f1, f2, f3]


#J_A = sp.Matrix(functions_A).jacobian(variables)
J_B = sp.Matrix(functions_B).jacobian(variables)
print("Rank of Jacobian for functions A:", J_A.rank())
print("Rank of Jacobian for functions B:", J_B.rank())

In [ ]:
grad_f3 = sp.Matrix([f3]).jacobian(variables)
J_B = J_A.row_insert(J_A.rows, grad_f3)

In [ ]:
u, relations, G = groebner_relations_among(functions_A, variables)

print("u variables:")
print(u)

print("Relations:")
for rel in relations:
    sp.pprint(rel)

In [ ]:
# Remove the unnecessary variables 
u, relations, G = groebner_relations_among(functions_B, variables)

print("u variables:")
print(u)

print("Relations:")
for rel in relations:
    sp.pprint(rel)

In [ ]:
import sympy as sp

x, y = sp.symbols('x y')

F = sp.Matrix([
    x**2 + y,
    x*y
])

J = F.jacobian([x, y])

In [ ]:
detJ = sp.factor(J.det())

In [ ]:
critical = sp.solve(detJ, [y])

In [ ]:
critical

In [ ]:
# Keep f1 and f2 constant
A = sp.Matrix([f1, f2]).jacobian(q)

In [ ]:
Q_variables = variables
T_variables = variables
J_Q = sp.simplify(sp.Matrix([f1, f2, f3, f4, f5, fc]).jacobian(variables))
J_Qf = sp.simplify(sp.Matrix([f1, f2, f3, f4, f5, fx]).jacobian(variables))

In [ ]:
 # Manifold space 
A = sp.simplify(sp.Matrix([f1, f2, f3, f4, f5]).jacobian(variables))
basis = A.nullspace()
N = sp.Matrix.hstack(*basis)

In [ ]:
J_Q_r = sp.simplify(J_Q * N)
J_Qf_r = sp.simplify(J_Qf * N)


In [ ]:
# Only study how f5 changes
J_f5_restricted = sp.simplify(
    sp.Matrix([f5]).jacobian(variables) * N
)

J_fc_restricted = sp.simplify(
    sp.Matrix([fc]).jacobian(variables) * N
)

In [ ]:
J_Q_r.rank()

In [ ]:
J_Qf_r.rank()

In [ ]:
J_f5_restricted

In [ ]:
J_f4